## Invoice DocAI Project (v2) — 01 Prepare SROIE

**Goal:** validate dataset structure and build train/val manifests.

Steps:
1. Mount Drive / resolve project root
2. Check SROIE dataset structure
3. Build manifests with `build_manifest()`
4. Save to `data/sroie/processed/`
5. Quality checks

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Mount Google Drive in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

def is_project_root(p: Path) -> bool:
    return (
        (p / 'data' / 'sroie').exists()
        and ((p / 'notebooks').exists() or (p / 'v2' / 'notebooks').exists())
    )

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd, cwd.parent, cwd.parent.parent,
        cwd / 'invoice_docai',
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path(r'c:\Yandex.Disk\Yandex.Disk\ML Neimark\From OCR 2 Transformers\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
    raise FileNotFoundError('Cannot locate invoice_docai project root')

PROJECT_ROOT = resolve_project_root()

# v2 src
V2_SRC = PROJECT_ROOT / 'v2' / 'src'
if V2_SRC.exists() and str(V2_SRC) not in sys.path:
    sys.path.insert(0, str(V2_SRC))
elif (PROJECT_ROOT / 'src').exists() and str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from docai_utils import build_manifest, normalize_date, normalize_total

DATA_ROOT = PROJECT_ROOT / 'data' / 'sroie'
RAW_ROOT = DATA_ROOT / 'raw'
PROCESSED_ROOT = DATA_ROOT / 'processed'
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT  = {PROJECT_ROOT}')
print(f'DATA_ROOT     = {DATA_ROOT}')
print(f'RAW_ROOT      = {RAW_ROOT}')
print(f'PROCESSED_ROOT= {PROCESSED_ROOT}')

## 1. Check SROIE dataset structure

In [ ]:
# Detect dataset layout
sroie_dir = RAW_ROOT / 'SROIE2019'
if not sroie_dir.exists():
    sroie_dir = RAW_ROOT  # fallback

# Try common SROIE structures
if (sroie_dir / 'train' / 'img').exists():
    TRAIN_IMAGES = sroie_dir / 'train' / 'img'
    TRAIN_LABELS = sroie_dir / 'train' / 'entities'
    VAL_IMAGES   = sroie_dir / 'test' / 'img'
    VAL_LABELS   = sroie_dir / 'test' / 'entities'
elif (sroie_dir / 'task3' / 'train').exists():
    TRAIN_IMAGES = sroie_dir / 'task3' / 'train' / 'img'
    TRAIN_LABELS = sroie_dir / 'task3' / 'train' / 'gt'
    VAL_IMAGES   = sroie_dir / 'task3' / 'test' / 'img'
    VAL_LABELS   = sroie_dir / 'task3' / 'test' / 'gt'
else:
    raise FileNotFoundError(f'Unknown SROIE layout in {sroie_dir}')

train_imgs = sorted(TRAIN_IMAGES.glob('*.jpg'))
val_imgs   = sorted(VAL_IMAGES.glob('*.jpg'))
print(f'Train images: {len(train_imgs)}')
print(f'Val images:   {len(val_imgs)}')
print(f'Train labels dir: {TRAIN_LABELS} (exists={TRAIN_LABELS.exists()})')
print(f'Val labels dir:   {VAL_LABELS} (exists={VAL_LABELS.exists()})')

## 2. Show sample images

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
rng = np.random.RandomState(42)
sample_ids = rng.choice(len(train_imgs), size=min(6, len(train_imgs)), replace=False)

for ax, idx in zip(axes.flat, sample_ids):
    img = Image.open(train_imgs[idx])
    ax.imshow(img)
    ax.set_title(train_imgs[idx].stem, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample SROIE Train Images', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build manifests

In [ ]:
manifest_train = build_manifest(TRAIN_IMAGES, TRAIN_LABELS)
manifest_val   = build_manifest(VAL_IMAGES, VAL_LABELS)

print(f'Train manifest: {manifest_train.shape}')
print(f'Val manifest:   {manifest_val.shape}')
display(manifest_train.head(3))
display(manifest_val.head(3))

## 4. Save manifests

In [ ]:
# Save CSV
manifest_train.to_csv(PROCESSED_ROOT / 'manifest_train.csv', index=False)
manifest_val.to_csv(PROCESSED_ROOT / 'manifest_val.csv', index=False)

# Save JSONL
manifest_train.to_json(PROCESSED_ROOT / 'manifest_train.jsonl', orient='records', lines=True, force_ascii=False)
manifest_val.to_json(PROCESSED_ROOT / 'manifest_val.jsonl', orient='records', lines=True, force_ascii=False)

for name in ['manifest_train.csv', 'manifest_train.jsonl', 'manifest_val.csv', 'manifest_val.jsonl']:
    path = PROCESSED_ROOT / name
    print(f'  {name}: {path.stat().st_size:,} bytes')

## 5. Quality checks

In [ ]:
for split_name, df in [('train', manifest_train), ('val', manifest_val)]:
    print(f'\n=== {split_name} ({len(df)} docs) ===')
    for field in ['gt_vendor', 'gt_date', 'gt_total']:
        empty = (df[field].fillna('').str.strip() == '').sum()
        print(f'  {field}: empty {empty}/{len(df)} ({empty/len(df)*100:.1f}%)')

## Done

Manifests saved to `data/sroie/processed/`. Proceed to notebook 02 (OCR baseline) or 03 (Donut).